# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:


# Load data from the 'data' subdirectory
# Using ISO-8859-1 encoding to handle special characters common in aviation datasets
df = pd.read_csv('data/AviationData.csv', encoding='ISO-8859-1')

# 1. Inspect Data Types and Non-Null Counts
print("--- Dataframe Info ---")
print(df.info())

# 2. Inspect Missing Values (NaNs)
print("\n--- Missing Values Count ---")
print(df.isna().sum())

# 3. View Summary Statistics
print("\n--- Summary Statistics ---")
display(df.describe())

C:\Users\STRATH\AppData\Local\Temp\ipykernel_28736\3234169906.py:3: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv', encoding='ISO-8859-1')


--- Dataframe Info ---
<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [3]:
# df.columns[[6, 7, 28]]

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [5]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
# inspecting column 6
df[df.columns[6]].unique()[:20]

array([nan, 36.922223, 42.445277, 30.757778, 46.041111, 48.12, 38.54,
       46.154444, 70.333333, 30.383611, '38.335', '45.928334',
       '29.607222', '35.389722', '33.948611', '64.8', '41.908889',
       '47.852222', '33.938611', '38.981667'], dtype=object)

In [7]:
# converting to Numeric
df[df.columns[6]] = pd.to_numeric(df[df.columns[6]], errors='coerce')
df[df.columns[7]] = pd.to_numeric(df[df.columns[7]], errors='coerce')
df[df.columns[28]] = pd.to_numeric(df[df.columns[28]], errors='coerce')

In [8]:
# inspecting column 6 again
df[df.columns[6]].unique()[:20]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                11986 non-null  float64
 7   Longitude               11974 non-null  float64
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [9]:
#pd.set_option('display.max_columns', None)
#df.head()

In [10]:
# filter columns to keep
cols_to_keep = [
    'Event.Date', 'Make', 'Model',
    'Aircraft.damage',
    'Total.Fatal.Injuries', 'Total.Serious.Injuries',
    'Total.Minor.Injuries', 'Total.Uninjured',
    'Amateur.Built', 'Aircraft.Category',
    'Weather.Condition', 'Broad.phase.of.flight',
    'Purpose.of.flight', 'Air.carrier','Engine.Type', 'Number.of.Engines'
]

df = df[cols_to_keep]
df.head()

,Event.Date,Make,Model,Aircraft.damage,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Amateur.Built,Aircraft.Category,Weather.Condition,Broad.phase.of.flight,Purpose.of.flight,Air.carrier,Engine.Type,Number.of.Engines
0,1948-10-24,Stinson,108-3,Destroyed,2.0,0.0,0.0,0.0,No,NaN,UNK,NaN,Personal,NaN,Reciprocating,1.0
1,1962-07-19,Piper,PA24-180,Destroyed,4.0,0.0,0.0,0.0,No,NaN,UNK,NaN,Personal,NaN,Reciprocating,1.0
2,1974-08-30,Cessna,172M,Destroyed,3.0,NaN,NaN,NaN,No,NaN,IMC,NaN,Personal,NaN,Reciprocating,1.0
3,1977-06-19,Rockwell,112,Destroyed,2.0,0.0,0.0,0.0,No,NaN,IMC,NaN,Personal,NaN,Reciprocating,1.0
4,1979-08-02,Cessna,501,Destroyed,1.0,2.0,NaN,0.0,No,NaN,VMC,NaN,Personal,NaN,NaN,NaN


In [11]:
# convert to datetime data type
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

In [12]:
# Convert injury columns to numeric
injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [13]:
# drop rows that have mssing date on critical columns
df = df.dropna(subset=['Make', 'Model', 'Event.Date'])

In [14]:
# check missing values per column
df.isna().sum().sort_values(ascending=False)

Broad.phase.of.flight     88777
Air.carrier               72168
Aircraft.Category         56532
Total.Serious.Injuries    12490
Total.Minor.Injuries      11914
Total.Fatal.Injuries      11386
Engine.Type                7025
Purpose.of.flight          6138
Number.of.Engines          6023
Total.Uninjured            5897
Weather.Condition          4439
Aircraft.damage            3172
Amateur.Built                99
Make                          0
Model                         0
Event.Date                    0
dtype: int64

In [15]:
# filter dats from 1983
df = df[df['Event.Date'].dt.year >= 1983]

In [16]:
# remove amateur-built aircraft
df = df[df['Amateur.Built'] == 'No']

In [17]:
#df.sample(10)

In [18]:
# filter aircraft category, include Airplanes and NAs as NAs are most likely airplanes
df = df[
    (df['Aircraft.Category'] == 'Airplane') |
    (df['Aircraft.Category'].isna())
]


In [19]:
# Check values in Aircraft.Category column
df['Aircraft.Category'].value_counts(dropna = False)

Aircraft.Category
NaN         51503
Airplane    21428
Name: count, dtype: int64

In [20]:
#Check duplicates
df.duplicated().sum()

np.int64(142)

In [21]:
# drop duplicates
df = df.drop_duplicates()

In [22]:
#df.duplicated().sum()

In [23]:
df['Aircraft.Category'].value_counts(dropna=False)

Aircraft.Category
NaN         51379
Airplane    21410
Name: count, dtype: int64

In [24]:
# final checks
df.shape

(72789, 16)

In [25]:
df.info()

<class 'pandas.DataFrame'>
Index: 72789 entries, 3600 to 88888
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Date              72789 non-null  datetime64[us]
 1   Make                    72789 non-null  str           
 2   Model                   72789 non-null  str           
 3   Aircraft.damage         69888 non-null  str           
 4   Total.Fatal.Injuries    63178 non-null  float64       
 5   Total.Serious.Injuries  62214 non-null  float64       
 6   Total.Minor.Injuries    62703 non-null  float64       
 7   Total.Uninjured         68089 non-null  float64       
 8   Amateur.Built           72789 non-null  str           
 9   Aircraft.Category       21410 non-null  str           
 10  Weather.Condition       69090 non-null  str           
 11  Broad.phase.of.flight   0 non-null      float64       
 12  Purpose.of.flight       67483 non-null  str           
 13 

In [26]:
df.sample(10)

,Event.Date,Make,Model,Aircraft.damage,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Amateur.Built,Aircraft.Category,Weather.Condition,Broad.phase.of.flight,Purpose.of.flight,Air.carrier,Engine.Type,Number.of.Engines
41778,1997-05-14,Hiller,UH-12E,Substantial,0.0,1.0,0.0,0.0,No,NaN,VMC,NaN,Positioning,NaN,Turbo Shaft,1.0
24549,1989-11-27,Piper,PA-23-160,Substantial,0.0,0.0,0.0,3.0,No,NaN,VMC,NaN,Instructional,NaN,Reciprocating,2.0
47994,2000-03-15,Cessna,402C,Substantial,0.0,0.0,0.0,1.0,No,NaN,VMC,NaN,Unknown,(dba: National Express),Reciprocating,2.0
36667,1995-01-19,Cessna,150M,Substantial,0.0,0.0,0.0,1.0,No,NaN,VMC,NaN,Personal,NaN,Reciprocating,1.0
80420,2017-08-08,EMBRAER,EMB135,NaN,0.0,0.0,0.0,26.0,No,Airplane,NaN,NaN,NaN,JetGo,NaN,2.0
38103,1995-08-18,Beech,35N,Substantial,0.0,0.0,0.0,2.0,No,NaN,VMC,NaN,Personal,NaN,Reciprocating,1.0
16649,1987-01-15,Piper,PA-28-181,Substantial,0.0,0.0,0.0,2.0,No,NaN,IMC,NaN,Personal,NaN,Reciprocating,1.0
13260,1985-10-16,Mitsubishi,MU-2B-2D,Destroyed,1.0,0.0,0.0,0.0,No,NaN,IMC,NaN,Unknown,Air Exchange Inc.,Turbo Prop,2.0
23546,1989-07-20,Hiller,UH-12E,Destroyed,0.0,0.0,1.0,0.0,No,NaN,VMC,NaN,Aerial Application,NaN,Reciprocating,1.0
71902,2012-06-03,CESSNA,150G,Substantial,0.0,0.0,0.0,2.0,No,Airplane,VMC,NaN,Personal,NaN,Reciprocating,1.0


**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

In [27]:
# Create new column "total.people" on board
df['Total.People'] = (
    df['Total.Fatal.Injuries'].fillna(0) +
    df['Total.Serious.Injuries'].fillna(0) +
    df['Total.Minor.Injuries'].fillna(0) +
    df['Total.Uninjured'].fillna(0)
)

In [28]:
# Create new column Fatality rate
df['Fatality.Rate'] = (
    df['Total.Fatal.Injuries'].fillna(0) / df['Total.People']
)

In [29]:
# Create new column Serious injury rate
df['Serious.Injury.Rate'] = df['Total.Serious.Injuries'] / df['Total.People']

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [30]:
# Create destruction flag
df['Destroyed'] = df['Aircraft.damage'].apply(
    lambda x: 1 if x == 'Destroyed' else 0
)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [31]:
df['Make'].value_counts().head(20)

Make
Cessna               20733
Piper                11246
CESSNA                4914
Beech                 4052
PIPER                 2838
Bell                  1780
Boeing                1493
BOEING                1139
BEECH                 1041
Mooney                1029
Grumman                978
Bellanca               819
Hughes                 685
Robinson               656
Air Tractor            576
Schweizer              544
Mcdonnell Douglas      506
Aeronca                458
Maule                  427
Champion               413
Name: count, dtype: int64

In [32]:
#df['Make'].nunique()

In [33]:
df['Make'].dropna().sort_values().unique()[:50]

<StringArray>
[                     '177MF LLC',                    '2021FX3 LLC',
                         '3XTRIM',                            '737',
                            '777',               'AAA AIRCRAFT LLC',
           'AB Sportine Aviacija',             'ADAMS DENNIS ALLEN',
                      'AERMACCHI',                 'AERO ADVENTURE',
                 'AERO AT SP ZOO',                 'AERO COMMANDER',
                  'AERO SP Z O O',                 'AERO VODOCHODY',
                    'AEROFAB INC',                   'AEROFAB INC.',
                        'AEROMOT',                        'AERONCA',
                      'AEROPRAKT', 'AEROPRAKT MANUFACTURING SP ZOO',
                     'AEROPRO CZ',               'AEROPRO CZ S R O',
                      'AEROS LTD',   'AEROS LTD/SKYRANGER AIRCRAFT',
                   'AEROSPATIALE',            'AEROSPATIALE ALENIA',
                      'AEROSPOOL',                  'AEROSPORT LTD',
    'AEROSTAR ACFT C

In [34]:
# remove any trailing white space if any and make it uppercase
df['Make'] = df['Make'].str.strip().str.upper()

In [35]:
# drop nulls
df = df.dropna(subset=['Make'])

In [36]:
# Check frequency counts
make_counts = df['Make'].value_counts()
make_counts.head(20)

Make
CESSNA               25647
PIPER                14084
BEECH                 5093
BOEING                2632
BELL                  1793
MOONEY                1270
GRUMMAN               1055
BELLANCA               978
HUGHES                 686
ROBINSON               673
AIR TRACTOR            672
AERONCA                606
MCDONNELL DOUGLAS      576
MAULE                  571
SCHWEIZER              549
CHAMPION               504
STINSON                421
AERO COMMANDER         409
DE HAVILLAND           403
LUSCOMBE               390
Name: count, dtype: int64

In [37]:
# Keep only meaningful Makes (threshold = 50)
valid_makes = make_counts[make_counts >= 50].index

df = df[df['Make'].isin(valid_makes)]

 Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [38]:
df['Model'].isna().sum()

np.int64(0)

In [39]:
df['Model'].value_counts().head(20)

Model
152          2176
172          1643
172N         1090
PA-28-140     860
172M          757
150           750
172P          664
182           617
180           594
150M          550
PA-18-150     549
PA-18         549
PA-28-180     546
PA-28-161     523
PA-28-181     508
737           484
150L          435
A36           428
PA-38-112     423
G-164A        417
Name: count, dtype: int64

In [40]:
df.groupby('Make')['Model'].nunique().sort_values(ascending=False).head(20)

Make
CESSNA               816
PIPER                640
BOEING               547
BEECH                533
BELL                 233
MCDONNELL DOUGLAS    135
AEROSPATIALE         115
NORTH AMERICAN       113
SCHWEIZER            113
MAULE                111
DE HAVILLAND         108
GRUMMAN              106
DOUGLAS               93
AERO COMMANDER        92
EMBRAER               90
AERONCA               86
BELLANCA              82
AIRBUS INDUSTRIE      78
ROCKWELL              76
MOONEY                74
Name: Model, dtype: int64

In [41]:
# Create derived column
df['Aircraft.Type'] = df['Make'].str.strip() + " " + df['Model'].str.strip()

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Note**: You do not necessarily need to impute or drop NaNs here.

In [42]:
df['Engine.Type'].value_counts(dropna=False)

Engine.Type
Reciprocating      53795
NaN                 4560
Turbo Prop          2794
Turbo Fan           2169
Turbo Shaft         2077
Unknown             1451
Turbo Jet            537
Geared Turbofan       12
UNK                    1
Name: count, dtype: int64

In [43]:
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()

In [44]:
# Weather condition
df['Weather.Condition'].value_counts(dropna=False)

Weather.Condition
VMC    58138
IMC     5172
NaN     3195
UNK      696
Unk      195
Name: count, dtype: int64

In [45]:
# clean
df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()

In [46]:
#Recheck after cleaning
df['Weather.Condition'].value_counts(dropna=False)

Weather.Condition
VMC    58138
IMC     5172
NaN     3195
UNK      891
Name: count, dtype: int64

In [47]:
# for NaN make it unknown
df['Weather.Condition'] = df['Weather.Condition'].fillna('UNK')

In [48]:
df['Weather.Condition'].value_counts(dropna=False)

Weather.Condition
VMC    58138
IMC     5172
UNK     4086
Name: count, dtype: int64

In [49]:
df['Number.of.Engines'].value_counts(dropna=False)

Number.of.Engines
1.0    52349
2.0     9435
NaN     4266
0.0      517
3.0      430
4.0      399
Name: count, dtype: int64

In [50]:
# drop rows missing number of enginees
df = df.dropna(subset=['Number.of.Engines'])

In [51]:
# inspect purpose of flight
df['Purpose.of.flight'].value_counts(dropna=False)

Purpose.of.flight
Personal                     35255
Instructional                 8593
Unknown                       4868
Aerial Application            3788
Business                      3221
NaN                           2581
Positioning                   1249
Other Work Use                 855
Ferry                          601
Public Aircraft                590
Aerial Observation             548
Executive/corporate            387
Skydiving                      170
Flight Test                    129
Banner Tow                      96
Public Aircraft - Federal       48
Glider Tow                      33
Public Aircraft - State         29
Firefighting                    19
Public Aircraft - Local         18
Air Race/show                   16
Air Race show                   15
External Load                   10
Air Drop                         5
PUBS                             3
ASHO                             3
Name: count, dtype: int64

In [52]:
# standardise text
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()

In [53]:
# inspect Broad.phase.of.flight
df['Broad.phase.of.flight'].value_counts(dropna=False)

Broad.phase.of.flight
NaN    63130
Name: count, dtype: int64

In [54]:
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].astype(str).str.strip().str.title()

In [55]:
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')

In [56]:
df['Broad.phase.of.flight'].value_counts(dropna=False)

Broad.phase.of.flight
Unknown    63130
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [57]:
missing_pct = (df.isna().sum() / len(df)) * 100
missing_pct.sort_values(ascending=False)

Air.carrier               83.347062
Aircraft.Category         73.926818
Serious.Injury.Rate       14.432124
Total.Serious.Injuries    14.032948
Total.Minor.Injuries      13.206083
Total.Fatal.Injuries      13.101537
Total.Uninjured            5.707271
Purpose.of.flight          4.088389
Engine.Type                2.710280
Aircraft.damage            2.605734
Fatality.Rate              0.470458
Event.Date                 0.000000
Model                      0.000000
Make                       0.000000
Broad.phase.of.flight      0.000000
Amateur.Built              0.000000
Weather.Condition          0.000000
Total.People               0.000000
Number.of.Engines          0.000000
Destroyed                  0.000000
Aircraft.Type              0.000000
dtype: float64

In [58]:
# drop columns with many nulls
cols_to_drop = [
    'Air.carrier',
    'Aircraft.Category'
]

df = df.drop(columns=cols_to_drop)



In [59]:
df.head()

,Event.Date,Make,Model,Aircraft.damage,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Amateur.Built,Weather.Condition,Broad.phase.of.flight,Purpose.of.flight,Engine.Type,Number.of.Engines,Total.People,Fatality.Rate,Serious.Injury.Rate,Destroyed,Aircraft.Type
3601,1983-01-01,CESSNA,182P,Substantial,0.0,0.0,1.0,3.0,No,VMC,Unknown,Personal,Reciprocating,1.0,4.0,0.0,0.0,0,CESSNA 182P
3602,1983-01-01,CESSNA,182RG,Substantial,0.0,0.0,0.0,2.0,No,VMC,Unknown,Personal,Reciprocating,1.0,2.0,0.0,0.0,0,CESSNA 182RG
3603,1983-01-01,CESSNA,182P,Substantial,0.0,0.0,0.0,1.0,No,VMC,Unknown,Personal,Reciprocating,1.0,1.0,0.0,0.0,0,CESSNA 182P
3604,1983-01-01,PIPER,PA-28R-200,Substantial,0.0,0.0,2.0,0.0,No,VMC,Unknown,Personal,Reciprocating,1.0,2.0,0.0,0.0,0,PIPER PA-28R-200
3605,1983-01-01,CESSNA,140,Substantial,0.0,0.0,0.0,2.0,No,VMC,Unknown,Instructional,Reciprocating,1.0,2.0,0.0,0.0,0,CESSNA 140


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [60]:
df.to_csv('cleaned_aviation_data.csv', index=False)